# Formative Assessment - 01

## Exercise: Outlier Detection and Removal using Quantile, Standard Deviation, Z-Score and IQR Methods

**Course Code:** 125MIH020  
**Student Name:** __________________________  
**Roll No.:** ______________________________

### Problem Statement
As a data analyst, identify and handle potential outliers in the `test_score` column. Outliers may represent exceptional student performance, lack of participation, or misunderstanding of the test material.

### Objectives
1. Visualize the test-score distribution.
2. Detect outliers using the Quantile/Percentile method.
3. Remove outliers using the Standard Deviation method.
4. Calculate and use Z-scores manually.
5. Detect and remove outliers using the IQR method.
6. Compare the original and cleaned data.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [2]:
df=pd.read_csv("student_test_scores.csv")


In [3]:
df.head()


  student_id  test_score
0    STU_001        77.0
1    STU_002        70.6
2    STU_003        78.5
3    STU_004        87.2
4    STU_005        69.7

In [4]:
df.describe()


       test_score
count  205.000000
mean    73.923902
std     22.485391
min      5.000000
25%     64.900000
50%     72.100000
75%     78.600000
max    220.000000

## 1. Visualize the Original Data

A box plot helps identify extreme observations, while a histogram shows the overall distribution of test scores.


In [5]:
plt.figure(figsize=(10,6))
sns.boxplot(x=df['test_score'])
plt.title('Distribution of Test Scores - Original Data')
plt.xlabel('Test Scores')
plt.show()

plt.figure(figsize=(10,6))
plt.hist(df['test_score'], bins=20,rwidth=0.8)
plt.xlabel('Test Scores')
plt.ylabel('Frequency')
plt.title('Histogram of Test Scores - Original Data')
plt.show()


## 2. Quantile / Percentile Method

The lower and upper thresholds are obtained from the 2nd and 98th percentiles. Values outside this range are treated as outliers.


In [6]:
lower_threshold = df['test_score'].quantile(0.02)
upper_threshold = df['test_score'].quantile(0.98)

print(lower_threshold)
print(upper_threshold)


26.664
148.7999999999998


In [7]:
df_quantile_cleaned = df[(df['test_score'] >= lower_threshold) & (df['test_score'] <= upper_threshold)]
df_quantile_cleaned.head()
df.shape, df_quantile_cleaned.shape


((205, 2), (195, 2))

In [8]:
plt.figure(figsize=(10,6))

plt.subplot(1,2,1)
sns.boxplot(x=df['test_score'])
plt.title('Original Data')

plt.subplot(1,2,2)
sns.boxplot(x=df_quantile_cleaned['test_score'])
plt.title('After Quantile Method')

plt.show()


## 3. Standard Deviation Method

The mean and standard deviation are calculated after percentile cleaning. A threshold of 3 standard deviations is used.

Lower limit = Mean - 3 × Standard Deviation  
Upper limit = Mean + 3 × Standard Deviation


In [9]:
mean_score = df_quantile_cleaned['test_score'].mean()
std_score = df_quantile_cleaned['test_score'].std()

print(f"Mean Test Score: {mean_score}, Standard Deviation: {std_score}")


Mean Test Score: 72.66358974358974, Standard Deviation: 11.390639014138308


In [10]:
lower_limit = mean_score - 3 * std_score
upper_limit = mean_score + 3 * std_score

print(f"Lower Limit: {lower_limit}, Upper Limit: {upper_limit}")


Lower Limit: 38.491672701174814, Upper Limit: 106.83550678600466


In [11]:
df_std_cleaned = df_quantile_cleaned[(df_quantile_cleaned['test_score'] >= lower_limit) & (df_quantile_cleaned['test_score'] <= upper_limit)]
df_std_cleaned.head()
df_quantile_cleaned.shape, df_std_cleaned.shape


((195, 2), (193, 2))

In [12]:
plt.figure(figsize=(10,6))

plt.subplot(1,2,1)
plt.hist(df_quantile_cleaned['test_score'], bins=20,rwidth=0.8)
plt.title('Before Standard Deviation Method')
plt.xlabel('Test Scores')
plt.ylabel('Frequency')

plt.subplot(1,2,2)
plt.hist(df_std_cleaned['test_score'], bins=20,rwidth=0.8)
plt.title('After Standard Deviation Method')
plt.xlabel('Test Scores')
plt.ylabel('Frequency')

plt.show()


## 4. Z-Score Method

The Z-score is calculated manually using:

**Z = (x - mean) / standard deviation**

A value is considered an outlier when `|Z| > 3`.

The Z-score method is equivalent to the 3-standard-deviation method because a Z-score of +3 or -3 means the observation is exactly three standard deviations above or below the mean.


In [13]:
df_zscore = df_std_cleaned.copy()

df_zscore['z_score'] = (df_zscore['test_score'] - df_zscore['test_score'].mean()) / df_zscore['test_score'].std()

df_zscore.head()


In [14]:
outliers_zscore = df_zscore[(df_zscore['z_score'] < -3) | (df_zscore['z_score'] > 3)]
outliers_zscore


In [15]:
df_zscore_cleaned = df_zscore[(df_zscore['z_score'] >= -3) & (df_zscore['z_score'] <= 3)]

df_zscore.shape, df_zscore_cleaned.shape


((193, 3), (193, 3))

In [16]:
plt.figure(figsize=(10,6))
sns.boxplot(x=df_zscore_cleaned['test_score'])
plt.title('Distribution of Test Scores - After Z-Score Method')
plt.xlabel('Test Scores')
plt.show()


## 5. IQR Method

The Interquartile Range method uses the middle 50% of the data.

IQR = Q3 - Q1

Lower limit = Q1 - 1.5 × IQR  
Upper limit = Q3 + 1.5 × IQR

Values outside these limits are treated as outliers.


In [17]:
Q1 = df_zscore_cleaned['test_score'].quantile(0.25)
Q3 = df_zscore_cleaned['test_score'].quantile(0.75)
IQR = Q3 - Q1

lower_limit_iqr = Q1 - 1.5 * IQR
upper_limit_iqr = Q3 + 1.5 * IQR

print(f"Q1: {Q1}, Q3: {Q3}")
print(f"IQR: {IQR}")
print(f"Lower Limit: {lower_limit_iqr}, Upper Limit: {upper_limit_iqr}")


Q1: 65.2, Q3: 77.9
IQR: 12.700000000000003
Lower Limit: 46.15, Upper Limit: 96.95000000000002


In [18]:
df_iqr_cleaned = df_zscore_cleaned[(df_zscore_cleaned['test_score'] >= lower_limit_iqr) & (df_zscore_cleaned['test_score'] <= upper_limit_iqr)]

df_zscore_cleaned.shape, df_iqr_cleaned.shape


((193, 3), (188, 3))

In [19]:
plt.figure(figsize=(10,6))

plt.subplot(1,2,1)
sns.boxplot(x=df['test_score'])
plt.title('Original Data')

plt.subplot(1,2,2)
sns.boxplot(x=df_iqr_cleaned['test_score'])
plt.title('Final Cleaned Data')

plt.show()


## 6. Compare Original and Final Cleaned Data

The descriptive statistics are compared to understand how the outlier removal process affected the distribution.


In [20]:
print("Original Data:")
print(df['test_score'].describe())

print("\nFinal Cleaned Data:")
print(df_iqr_cleaned['test_score'].describe())


Original Data:
count    205.000000
mean      73.923902
std       22.485391
min        5.000000
25%       64.900000
50%       72.100000
75%       78.600000
max      220.000000

Final Cleaned Data:
count    188.000000
mean      71.661702
std        9.059200
min       52.100000
25%       65.150000
50%       71.800000
75%       77.125000
max       96.600000


In [21]:
plt.figure(figsize=(10,6))

plt.subplot(1,2,1)
plt.hist(df['test_score'], bins=20,rwidth=0.8)
plt.title('Original Test Score Distribution')
plt.xlabel('Test Scores')
plt.ylabel('Frequency')

plt.subplot(1,2,2)
plt.hist(df_iqr_cleaned['test_score'], bins=20,rwidth=0.8)
plt.title('Final Cleaned Test Score Distribution')
plt.xlabel('Test Scores')
plt.ylabel('Frequency')

plt.show()


## 7. Final Results and Conclusion

The dataset initially contained **205 records**.

- After Quantile Method: **195 records**
- After Standard Deviation Method: **193 records**
- After Z-Score Method: **193 records**
- After IQR Method: **188 records**

### Conclusion
Four outlier detection techniques were studied: Quantile/Percentile, Standard Deviation, Z-Score and IQR. Visualization and descriptive statistics were used before and after cleaning. Different methods can identify different types of extreme observations, so the choice of method should depend on the distribution and analytical objective.
